## Trasnformacion a gold

### Objetivo:

    - Obtener informacion de peliculas, el pais donde se realizo la grabacion y cual fue la productora
    - A partir de fecha de lanzamiento 2010 
    - ordenar de manera ascendente por el titulo de la pelicula

    - mostrar titulo, pesupuesto, ingresos obtenidos, tiempo de duracion y fecha de lanzamiento
    - nombre de pais
    - nombre de la productora

tables: movie, country, production_country, production_company, movie_company \
columns: title, budget, revenue, duration_time, release_date, country_name, company_name, \
add: created_date 

df -> results_country_prod_company

### Leer archivos

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date =dbutils.widgets.get("p_file_date")

In [0]:
# movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# countries_df = spark.read.parquet(f"{silver_folder_path}/countries")
countries_df = spark.read.table("movie_silver.countries")

In [0]:
# production_country_df = spark.read.parquet(f"{silver_folder_path}/productions_countries")
production_country_df = spark.read.table("movie_silver.productions_countries") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# production_companies_df = spark.read.parquet(f"{silver_folder_path}/productions_companies")
production_companies_df = spark.read.table("movie_silver.productions_companies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# movies_companies_df = spark.read.parquet(f"{silver_folder_path}/movies_companies")
movies_companies_df = spark.read.table("movie_silver.movies_companies") \
                    .filter(f"file_date = '{v_file_date}'")

### Join "country" y "production_country"

In [0]:
country_prod_country_df = countries_df.join(production_country_df, 
                                            countries_df.country_id == production_country_df.country_id, 
                                            "inner") \
                                            .select(countries_df.country_name, countries_df.country_id, production_country_df.movie_id)

### Join "production_company" y "movie_company"

In [0]:
prod_comp_mov_comp_df = production_companies_df.join(movies_companies_df,
                                                    production_companies_df.company_id == movies_companies_df.company_id,
                                                    "inner") \
                                                    .select(production_companies_df.company_name, production_companies_df.company_id, 
                                                            movies_companies_df.movie_id)

### Join "movies_df", "languages_mov_lan_df" y "genres_mov_lan_df"

#### - Filtrar las películas donde su fecha de lanzamiento sea mayor o igual a 2010

In [0]:
movies_filter_df = movies_df.filter("year_release_date >= 2010")

In [0]:
results_movies_country_company_df = movies_filter_df.join(country_prod_country_df,
                                            movies_filter_df.movie_id == country_prod_country_df.movie_id,
                                            "inner") \
                                                       .join(prod_comp_mov_comp_df,
                                                             movies_filter_df.movie_id == prod_comp_mov_comp_df.movie_id,
                                                             "inner")

#### - Agregar la columna "created_date"

In [0]:
from pyspark.sql.functions import lit

In [0]:
results_df = results_movies_country_company_df \
    .select(movies_filter_df.movie_id, "country_id", "company_id", "title", "budget", "revenue", "duration_time", "release_date", "country_name", "company_name") \
    .withColumn("created_date", lit(v_file_date))

#### - Ordernar por la columna "titulo" de manera ascendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.title.asc())

In [0]:
display(results_order_by_df)

### Escribir los datos en el datalake en formato "Parquet"

In [0]:
# overwrite_partition("movie_gold", "results_country_prod_company","created_date",v_file_date)

In [0]:
# results_order_by_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_country_prod_company")
# results_order_by_df.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_country_prod_company")

# results_order_by_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_country_prod_company")

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date=src.created_date'
merge_delta_lake(results_order_by_df, "movie_gold", "results_country_prod_company", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_country_prod_company

In [0]:
# display(spark.read.parquet(f"{gold_folder_path}/results_country_prod_company"))